In [4]:
import cv2
import numpy as np

img = cv2.imread("1.jpg")
display = img.copy()

# storing 4 points
points = []

def order_points(pts):
    pts = np.array(pts, dtype="float32")

    # sum of coordinates (x + y)
    s = pts.sum(axis=1)
    # difference (x - y)
    diff = np.diff(pts, axis=1)

    # identify corners based on geometry
    top_left = pts[np.argmin(s)]
    bottom_right = pts[np.argmax(s)]
    top_right = pts[np.argmin(diff)]
    bottom_left = pts[np.argmax(diff)]

    # return in the correct order
    return np.array([top_left, top_right, bottom_right, bottom_left], dtype="float32")

def click(event, x, y, flags, param):
    global display

    # when left mouse is clicked and less than 4 points selected
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])  # save point

        # draw point on display image
        cv2.circle(display, (x, y), 5, (0, 0, 255), -1)

        # label the point number
        cv2.putText(display, str(len(points)), (x + 10, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        # update window
        cv2.imshow("Select 4 Points", display)

# show image and set mouse callback
cv2.imshow("Select 4 Points", display)
cv2.setMouseCallback("Select 4 Points", click)

print("Click 4 corner points, then press any key.")

# wait for key press
cv2.waitKey(0)
cv2.destroyAllWindows()

# check if 4 points selected
if len(points) != 4:
    print("select 4 points.")
else:
    # order points correctly
    pts = order_points(points)

    # define output size
    w, h = 400, 400

    # destination rectangle (top-down view)
    dst = np.array([
        [0, 0],
        [w - 1, 0],
        [w - 1, h - 1],
        [0, h - 1]
    ], dtype="float32")

    # compute perspective transform matrix
    M = cv2.getPerspectiveTransform(pts, dst)

    # transformation
    warped = cv2.warpPerspective(img, M, (w, h))

    cv2.imshow("Original", img)
    cv2.imshow("Warped", warped)
    cv2.imwrite("output.jpg", warped)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

Click 4 corner points, then press any key.
